In [ ]:
# import os

# # Adjust path to wherever Spark is installed
# os.environ["SPARK_HOME"] = "/Users/marthala/mydata/00_learning/databrick-local-simulator/stack/tmp/spark"  # change this path to your actual Spark install
# os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"  # or output of `/usr/libexec/java_home`

# import findspark
# findspark.init() 


# /Users/marthala/.pyenv/versions/3.12.4/lib/python3.12/site-packages/pyspark/jars

# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName("YourAppName").getOrCreate()
# sc = spark.sparkContext
# conf = sc.getConf()
# print("jars" , conf.get("spark.jars", ""))
# print(spark.sparkContext._jvm.java.lang.System.getProperty("java.class.path"))

# conf = spark.sparkContex

# print(conf.get("spark.jars", ""))
# print(conf.get("spark.jars.packages", ""))
# print(conf.get("spark.driver.extraClassPath", ""))
# print(conf.get("spark.executor.extraClassPath", ""))




# spark = SparkSession.builder.appName("kafka consumer").getOrCreate()

# print("Hadoop version:", spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

# jars_java = spark.sparkContext._jsc.sc().listJars()
# num_jars = jars_java.size()
# jars_py = [str(jars_java.get(i)) for i in range(num_jars)]
# print(jars_py)


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, TimestampType, DoubleType

spark = SparkSession.builder \
    .appName("kafka consumer") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0") \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()


df =  spark \
        .readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:29092") \
        .option("subscribe", "orders") \
        .option("startingOffsets", "latest") \
        .load()

schema = StructType() \
    .add("OrderID", StringType()) \
    .add("CustomerID", StringType())


processed_df = df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*")

# Output to the console (for testing)
query = processed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

query.awaitTermination()





In [ ]:
sc = spark.sparkContext
conf = sc.getConf()
print("jars" , conf.get("spark.jars", ""))
print(spark.sparkContext._jvm.java.lang.System.getProperty("java.class.path"))

print(sc.sparkHome)

In [ ]:
"""
{
	"OrderID": 11078,
	"CustomerID": "VAFFE",
	"EmployeeID": "6",
	"OrderDate": "2025-06-21",
	"RequiredDate": "2025-06-27",
	"ShippedDate": "2025-06-22",
	"ShipVia": 1,
	"Freight": 160.64,
	"ShipName": "Figueroa and Sons",
	"ShipAddress": "18196 Anthony Forge",
	"ShipCity": "New Carolyn",
	"ShipRegion": "OH",
	"ShipPostalCode": "26563",
	"ShipCountry": "Saint Barthelemy",
	"OrderDetails": [
		{
			"OrderID": 11078,
			"ProductID": 3,
			"UnitPrice": 18.82,
			"Quantity": 18,
			"Discount": 0.15
		},
		{
			"OrderID": 11078,
			"ProductID": 58,
			"UnitPrice": 46.47,
			"Quantity": 4,
			"Discount": 0
		}
	]
}

"""

In [ ]:
schema = StructType() \
    .add("OrderID", StringType()) \
    .add("CustomerID", StringType())

In [ ]:
# spark = SparkSession.builder \
#     .appName("kafka consumer") \
#     .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0") \
#     .getOrCreate()

In [ ]:
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subsribe", "orders") \
    .option("startingOffsets", "latest") \
    .load()


In [ ]:
spark.stop()